<a href="https://colab.research.google.com/github/indrap23/PJJDA/blob/main/Semi_Supervised_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Semi-supervised learning digunakan ketika Anda memiliki sejumlah besar data, tetapi hanya sebagian kecil yang memiliki label kategori. Memberikan label pada data secara manual seringkali memakan waktu dan biaya.


Contoh Kasus: Kategorisasi Tiket Dukungan Pelanggan (Customer Support)
Sebuah perusahaan menerima 10.000 email keluhan per bulan. Untuk melatih sistem AI yang dapat mengkategorikan email ke divisi yang tepat ("Teknis", "Penagihan", "Umum"), algoritma membutuhkan contoh data historis.

Data Berlabel: Tim manusia hanya sempat mengkategorikan 500 email secara manual.

Data Tidak Berlabel: Sisa 9.500 email ada di dalam sistem, tetapi tidak memiliki label kategori.

Daripada hanya menggunakan 500 data untuk melatih model (Supervised Learning), atau membuang data berlabel untuk mencari pola sendiri (Unsupervised Learning), model semi-supervised menggunakan 500 data awal untuk mempelajari pola dasar, lalu model tersebut mencoba menebak label untuk 9.500 data sisanya. Tebakan yang paling meyakinkan (memiliki tingkat probabilitas tinggi) akan dijadikan label baru, dan model melatih dirinya sendiri kembali dengan kumpulan data yang kini lebih besar. Teknik ini sering disebut Self-Training atau Pseudo-labeling.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
# 1. Menyiapkan Data Simulasi
# Membuat 1000 data dengan 20 fitur (misal: representasi kata dalam email)
X, y = make_classification(n_samples=1000, n_features=20, n_classes=2, random_state=42)

# Memisahkan 20% data untuk pengujian akhir (test set)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
# 2. Mengatur Skenario Semi-Supervised
# Mensimulasikan kondisi di mana 90% data training tidak memiliki label.
# Dalam scikit-learn, data tidak berlabel direpresentasikan dengan angka -1.
rng = np.random.RandomState(42)
random_unlabeled_points = rng.rand(len(y_train)) < 0.90

y_train_semi = np.copy(y_train)
y_train_semi[random_unlabeled_points] = -1

jumlah_berlabel = len(y_train_semi[y_train_semi != -1])
jumlah_tidak_berlabel = len(y_train_semi[y_train_semi == -1])
print(f"Data berlabel: {jumlah_berlabel}")
print(f"Data tidak berlabel: {jumlah_tidak_berlabel}")

Data berlabel: 83
Data tidak berlabel: 717


In [5]:
# 3. Menyiapkan Model Dasar
# Menggunakan Random Forest sebagai model pengklasifikasi dasar
base_model = RandomForestClassifier(random_state=42)

In [6]:
# 4. Membangun Model Semi-Supervised
# Model akan melatih data berlabel, memprediksi yang tidak berlabel,
# dan memasukkan prediksi dengan keyakinan tinggi sebagai data training baru.
semi_supervised_model = SelfTrainingClassifier(base_model)
semi_supervised_model.fit(X_train, y_train_semi)

SelfTrainingClassifier(estimator=RandomForestClassifier(random_state=42))

In [7]:
# 5. Evaluasi Performa
y_pred = semi_supervised_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Akurasi model Semi-Supervised: {accuracy * 100:.2f}%")

Akurasi model Semi-Supervised: 83.00%
